# BTC 15M — सिर्फ कोड + UTC समय (तारीख हटाकर)
यह पहले से Google Drive में बनी **मूल PDF के पेज बिना दोबारा चार्ट बनाए** एक नए समय-आधारित समूह में जोड़ता है।
**उदाहरण:** `15RGG100014.pdf` से तारीख `14` हटेगी और वह `15RGG1000.pdf` समूह में जाएगी। बाकी तारीखों, महीनों और वर्षों का वही कोड व वही 10:00 UTC समय होने पर उनके सभी पेज भी इसी PDF में जुड़ेंगे। **मूल फाइलें और उनके अंदर की पूरी UTC तारीख/समय सुरक्षित रहेंगे।**
**पुराना फ़ोल्डर:** `MyDrive/BTC_15MIN_1H_2014_2026_V3_RESEARCH/ONE_15MIN_PDF_SET`
**नया एकमात्र टाइम-ओनली आउटपुट फ़ोल्डर:** `MyDrive/BTC_15MIN_1H_2014_2026_V3_RESEARCH/TIME_ONLY_15MIN_PDF_SET`
**सत्यापन:** पेज नहीं छूटने चाहिए; कुल मूल पेजों और नए PDF पेजों की तुलना होगी। पेज और PDF की सटीक संख्या `REPORTS/TIME_ONLY_FINAL_TOTALS.txt` में लिखी जाएगी। अधूरा काम उसी कोड से फिर चालू किया जा सकता है।


In [ ]:
# Bitcoin historical PDF regrouping: existing original 15-minute pages, TIME ONLY.
# READS existing PDFs. DOES NOT regenerate price charts or delete source files.
# Example: 15RGG100014.pdf + 15RGG100015.pdf -> 15RGG1000.pdf
# Group key = 15 + three candle colors + HHMM UTC; DAY omitted.

import sys, subprocess, re, csv, hashlib, shutil, os
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pypdf"])
    from pypdf import PdfReader, PdfWriter

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/BTC_15MIN_1H_2014_2026_V3_RESEARCH")
SOURCE = ROOT / "ONE_15MIN_PDF_SET"
DEST = ROOT / "TIME_ONLY_15MIN_PDF_SET"
REPORTS = ROOT / "REPORTS"
TEMP = Path("/content/btc_time_only_merge")

for folder in (DEST, REPORTS, TEMP):
    folder.mkdir(parents=True, exist_ok=True)

if not SOURCE.is_dir():
    raise RuntimeError("Original PDF folder was not found: " + str(SOURCE))

# Exact format: 15 + R/G three-letter code (or REVIEW) + HHMM + DD.
# Example: 15RGG100014.pdf -> 15RGG1000.pdf.
pattern = re.compile(
    r"^15(?P<code>[RG]{3}|REVIEW)(?P<hour>[01]\d|2[0-3])"
    r"(?P<minute>00|15|30|45)(?P<day>0[1-9]|[12]\d|3[01])\.pdf$",
    re.IGNORECASE
)

sources = sorted(SOURCE.glob("*.pdf"))
if not sources:
    raise RuntimeError(
        "No original PDFs found in: " + str(SOURCE)
        + ". Run the original 15M/1H generator first."
    )

groups = defaultdict(list)
unrecognized = []
for path in sources:
    match = pattern.fullmatch(path.name)
    if not match:
        unrecognized.append(path.name)
        continue
    time_code = (
        "15" + match["code"].upper()
        + match["hour"] + match["minute"] + ".pdf"
    )
    groups[time_code].append(path)

if unrecognized:
    rejected = REPORTS / "TIME_ONLY_UNRECOGNIZED_SOURCE_NAMES.txt"
    rejected.write_text("\n".join(unrecognized) + "\n", encoding="utf-8")
    raise RuntimeError(
        f"{len(unrecognized)} source PDF filenames have an unknown pattern. "
        f"To avoid missing data, regrouping stopped. See {rejected}."
    )

# Old counts report can speed up checking; only trust it while size+mtime match.
old_audit = REPORTS / "ACTUAL_PDF_PAGE_AUDIT.csv"
prior_counts = {}
if old_audit.exists():
    with old_audit.open("r", newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            prior_counts[row["filename"]] = row

def pages(pdf_path):
    return len(PdfReader(str(pdf_path), strict=False).pages)

def original_page_count(pdf_path):
    current = pdf_path.stat()
    old = prior_counts.get(pdf_path.name)
    if old and old.get("status") == "OK":
        try:
            if (int(old.get("bytes", "-1")) == current.st_size
                and int(old.get("modified_ns", "-1")) == current.st_mtime_ns):
                return int(old["pages"])
        except (ValueError, TypeError, KeyError):
            pass
    return pages(pdf_path)

# Safe to rerun: completed output groups stay untouched when source is identical.
manifest_path = REPORTS / "TIME_ONLY_MERGE_MANIFEST.csv"
previous = {}
if manifest_path.exists():
    with manifest_path.open("r", newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            previous[row["filename"]] = row

manifest_columns = ["filename", "source_fingerprint", "source_pdfs", "pages"]
completed = {}

def save_manifest():
    temp_csv = REPORTS / "TIME_ONLY_MERGE_MANIFEST.tmp.csv"
    with temp_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=manifest_columns)
        writer.writeheader()
        writer.writerows(
            completed[key] for key in sorted(completed)
        )
    temp_csv.replace(manifest_path)

print(f"Original 15M PDF files: {len(sources):,}")
print(f"Time-only PDF groups to build: {len(groups):,}")
print(f"Example: 15RGG100014.pdf -> 15RGG1000.pdf")
print("Original source files will NOT be deleted or modified.")
print("New Drive folder:", DEST)

total_original_pages = 0
total_new_pages = 0
reused = 0
rebuilt = 0

for index, (name, paths) in enumerate(sorted(groups.items()), 1):
    details = []
    expected_pages = 0
    for source in paths:
        n = original_page_count(source)
        expected_pages += n
        st = source.stat()
        details.append(f"{source.name}|{st.st_size}|{st.st_mtime_ns}|{n}")
    total_original_pages += expected_pages
    fingerprint = hashlib.sha256(
        "\n".join(details).encode("utf-8")
    ).hexdigest()

    output = DEST / name
    was = previous.get(name)
    reuse = (
        output.exists()
        and was is not None
        and was.get("source_fingerprint") == fingerprint
        and was.get("pages") == str(expected_pages)
    )
    if reuse:
        try:
            reuse = pages(output) == expected_pages
        except Exception:
            reuse = False

    if reuse:
        reused += 1
    else:
        local = TEMP / name
        writer = PdfWriter()
        try:
            for source in paths:
                writer.append(str(source), import_outline=False)
            with local.open("wb") as fh:
                writer.write(fh)
        finally:
            writer.close()

        actual = pages(local)
        if actual != expected_pages:
            raise RuntimeError(
                f"Page mismatch in {name}: expected {expected_pages}, got {actual}"
            )

        pending = DEST / (name + ".partial")
        shutil.copyfile(local, pending)

        if pages(pending) != expected_pages:
            raise RuntimeError("Drive copy failed page check: " + name)

        os.replace(pending, output)
        local.unlink(missing_ok=True)
        rebuilt += 1

    completed[name] = {
        "filename": name,
        "source_fingerprint": fingerprint,
        "source_pdfs": len(paths),
        "pages": expected_pages
    }
    total_new_pages += expected_pages

    if index % 20 == 0 or index == len(groups):
        save_manifest()
        print(
            f"Time-only PDFs: {index:,}/{len(groups):,}; "
            f"pages: {total_new_pages:,}; "
            f"new: {rebuilt:,}; reused: {reused:,}"
        )

if total_original_pages != total_new_pages:
    raise RuntimeError("PAGE INTEGRITY CHECK FAILED")

ranking = sorted(
    completed.values(),
    key=lambda x: (-x["pages"], x["filename"])
)

lines = [
    "ACTUAL TIME-ONLY BTC 15M PDF REPORT",
    "Generated UTC: " + datetime.now(timezone.utc).isoformat(),
    "Original PDF count: " + format(len(sources), ","),
    "Time-only PDF count: " + format(len(groups), ","),
    "Original total pages: " + format(total_original_pages, ","),
    "Time-only total pages: " + format(total_new_pages, ","),
    "ALL PAGES PRESERVED: YES",
    "PDFs created/rebuilt this run: " + format(rebuilt, ","),
    "PDFs reused this run: " + format(reused, ","),
    "Original source folder: " + str(SOURCE),
    "New time-only folder: " + str(DEST),
    "Largest PDF: " + ranking[0]["filename"],
    "Largest PDF pages: " + str(ranking[0]["pages"]),
    "",
    "TOP 10 PDFS BY PAGE COUNT:"
]
for number, item in enumerate(ranking[:10], 1):
    lines.append(
        f"{number}. {item['filename']} - {item['pages']} pages"
    )

report_path = REPORTS / "TIME_ONLY_FINAL_TOTALS.txt"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("\n" + "\n".join(lines))
print("\nReports:", report_path)
